In [ ]:
# Cell 2: Imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import os

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Dense, Dropout,
                                   BatchNormalization, GlobalAveragePooling2D,
                                   GaussianNoise, concatenate)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print("All imports successful!")

In [ ]:
# Cell 3: GPU Configuration & Dataset Check
tf.keras.backend.clear_session()

# Configure GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth configured")
    except RuntimeError as e:
        print(e)

# Check your dataset structure
def check_dataset_structure():
    train_path = "/train"
    val_path = "/val"
    test_path = "/test"

    print("=== CHECKING DATASET STRUCTURE ===")
    for split_path, split_name in [(train_path, 'TRAIN'), (val_path, 'VAL'), (test_path, 'TEST')]:
        print(f"\n{split_name} SET:")
        if os.path.exists(split_path):
            for class_name in os.listdir(split_path):
                class_path = os.path.join(split_path, class_name)
                if os.path.isdir(class_path):
                    num_images = len([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
                    print(f"  {class_name}: {num_images} images")
        else:
            print(f"  Path not found: {split_path}")

check_dataset_structure()

In [ ]:
# Cell 4: Configuration
IMG_SIZE = (384, 384)  # Higher resolution for better artifact detection
BATCH_SIZE = 16        # Smaller batches for higher resolution

print("Configuration:")
print(f"Image Size: {IMG_SIZE}")
print(f"Batch Size: {BATCH_SIZE}")

In [ ]:
# Cell 5: Data Generators & Class Weighting
train_path = "/train"
val_path = "/val"
test_path = "/test"

# Enhanced data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='constant',
    cval=0.0
)

val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Create data generators
train_flow = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_flow = val_datagen.flow_from_directory(
    val_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_flow = test_datagen.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# AGGRESSIVE class weighting to fix imbalance
print("\n=== COMPUTING CLASS WEIGHTS ===")
labels = train_flow.classes
class_weights_raw = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights_raw))

print("Class mapping:", train_flow.class_indices)
print("Aggressive class weights:", class_weights)

In [ ]:
# Cell 6: Simple Two-Stream Model
def create_simple_two_stream(input_shape=(384, 384, 3), num_classes=3):
    input_layer = Input(shape=input_shape)

    # Stream 1: Spatial Features (EfficientNet)
    spatial_base = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    spatial_base.trainable = False
    spatial_stream = spatial_base(input_layer)
    spatial_stream = GlobalAveragePooling2D()(spatial_stream)
    spatial_stream = Dense(256, activation='relu')(spatial_stream)
    spatial_stream = Dropout(0.5)(spatial_stream)

    # Stream 2: Noise Features (Simple CNN)
    noise_stream = Conv2D(32, (5,5), activation='relu')(input_layer)
    noise_stream = BatchNormalization()(noise_stream)
    noise_stream = MaxPooling2D(2,2)(noise_stream)
    noise_stream = Conv2D(64, (3,3), activation='relu')(noise_stream)
    noise_stream = GlobalAveragePooling2D()(noise_stream)
    noise_stream = Dense(128, activation='relu')(noise_stream)
    noise_stream = Dropout(0.4)(noise_stream)

    # Fusion
    combined = concatenate([spatial_stream, noise_stream])
    output = Dense(num_classes, activation='softmax')(combined)

    return Model(inputs=input_layer, outputs=output)

# Create model
model = create_simple_two_stream(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
model.summary()

In [ ]:
# Cell 7: Compile Model with Focal Loss
def focal_loss(gamma=2.0, alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)

        # Calculate focal loss
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * tf.pow(1. - y_pred, gamma)
        focal_loss = weight * cross_entropy

        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=1))
    return focal_loss_fixed

# Compile model
optimizer = Adam(learning_rate=0.0001)
model.compile(
    optimizer=optimizer,
    loss=focal_loss(),  # Better for imbalanced data
    metrics=['accuracy']
)

print("Model compiled with focal loss for imbalanced data!")

In [ ]:
# Cell 8: Training Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=12,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=6,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        '/two_stream_gan_detector_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

print("Callbacks configured!")

In [ ]:
# Cell 9: Phase 1 - Train Head Only
print("=== PHASE 1: Training Head Only (20 epochs) ===")

history1 = model.fit(
    train_flow,
    epochs=20,
    validation_data=val_flow,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)

print("Phase 1 completed!")

In [ ]:
# Cell 10: Save Model Properly

model.save('/phase1_two_stream_model.keras')
print("Model saved as: /phase1_two_stream_model.keras")

model.save_weights('/phase1_model_weights.weights.h5')
print("Weights saved as: /phase1_model_weights.weights.h5")

print("✅ Model saved successfully!")

In [ ]:
# Cell 11: Model Health Check
print("=== MODEL HEALTH CHECK ===")

# 1. random guessing baseline
print("Random guessing baseline for 3 classes: 33.33%")

# 2. current model performance
val_loss, val_accuracy = model.evaluate(val_flow, verbose=0)
print(f"Current validation accuracy: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)")

# 3. better than random?
if val_accuracy < 0.33:
    print("❌ ALERT: Model performing WORSE than random guessing!")
    print("   This means something is fundamentally broken")
else:
    print("✅ Model is learning (better than random)")

# 4. Quick prediction test
print("\n=== PREDICTION TEST ===")
test_batch, test_labels = next(iter(val_flow))
predictions = model.predict(test_batch[:3], verbose=0)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(test_labels[:3], axis=1)

print("Sample predictions shape:", predictions.shape)
print("Prediction probabilities:", predictions)
print("Predicted classes:", predicted_classes)
print("True classes:     ", true_classes)
print("Class names:      ", list(val_flow.class_indices.keys()))

In [ ]:
# Cell 17: Learning Curves & Progress Visualization
print("=== TRAINING PROGRESS VISUALIZATION ===")

# Plot learning curves
plt.figure(figsize=(15, 5))

# 1. Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history1.history['accuracy'], label='Train Accuracy', color='blue', alpha=0.7)
plt.plot(history1.history['val_accuracy'], label='Val Accuracy', color='red', alpha=0.7)
plt.title('Phase 1: Accuracy Progress')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# 2. Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history1.history['loss'], label='Train Loss', color='blue', alpha=0.7)
plt.plot(history1.history['val_loss'], label='Val Loss', color='red', alpha=0.7)
plt.title('Phase 1: Loss Progress')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print key statistics
print("\n📊 PHASE 1 SUMMARY:")
print(f"Final Training Accuracy: {history1.history['accuracy'][-1]:.4f} ({history1.history['accuracy'][-1]*100:.2f}%)")
print(f"Final Validation Accuracy: {history1.history['val_accuracy'][-1]:.4f} ({history1.history['val_accuracy'][-1]*100:.2f}%)")
print(f"Best Validation Accuracy: {max(history1.history['val_accuracy']):.4f} ({max(history1.history['val_accuracy'])*100:.2f}%)")

# Check for overfitting
train_final = history1.history['accuracy'][-1]
val_final = history1.history['val_accuracy'][-1]
gap = train_final - val_final

print(f"\n🔍 OVERFITTING ANALYSIS:")
print(f"Train-Val Gap: {gap:.4f}")
if gap > 0.1:
    print("✅ Good: Minimal overfitting")
else:
    print("✅ Excellent: Very little overfitting")

In [ ]:
# Cell 18: Current Performance
print("=== CURRENT MODEL PERFORMANCE ===")

test_batch, test_labels = next(iter(test_flow))
predictions = model.predict(test_batch[:8], verbose=0)

class_names = list(test_flow.class_indices.keys())

print("Sample Predictions (First 8 test images):")
print("TrueClass -> PredictedClass [Confidence]")
print("-" * 50)

for i in range(8):
    true_class = class_names[np.argmax(test_labels[i])]
    pred_class = class_names[np.argmax(predictions[i])]
    confidence = np.max(predictions[i])

    status = "✅ CORRECT" if true_class == pred_class else "❌ WRONG"
    print(f"{true_class:8} -> {pred_class:8} [{confidence:.2f}] {status}")

# current accuracy on test set
correct = sum([1 for i in range(8) if class_names[np.argmax(test_labels[i])] == class_names[np.argmax(predictions[i])]])
print(f"\n📈 Quick Test Accuracy: {correct}/8 = {correct/8*100:.1f}%")